# How readable are forward-looking statements?

The original run produced 100 filing rows and 21 columns.

**Running in Google Colab.** Set the filing, word-list, pronunciation-dictionary, and output paths. The inputs are SEC filing text files, PositiveWords.txt, NegativeWords.txt, LitigiousWords.txt, and cmudict.txt. Run the cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip -q install syllables
import os
import nltk
import pandas as pd
import syllables
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('punkt_tab')

In [ ]:
downloadDir = "/content/drive/MyDrive/TextMining/My10-Ks/"
wordListDir = "/content/drive/MyDrive/TextMining/WordLists/"
cmufile = "/content/drive/MyDrive/Fall 2026 Text Mining/week 04/cmudict.txt"
outputFile = "/content/drive/MyDrive/TextMining/Readability_results.csv"

complexThreshold = 3


In [ ]:
syllableDict = {}
with open(cmufile, encoding='latin-1') as cmu:
    for aline in cmu:
        if aline.startswith(';;;') or aline.startswith('#'):
            continue
        parts = aline.split()
        if len(parts) < 2:
            continue
        word = parts[0].lower().split('(')[0]
        syllableCount = 0
        for phoneme in parts[1:]:
            if phoneme[-1].isdigit():
                syllableCount += 1
        if syllableCount > 0 and word not in syllableDict:
            syllableDict[word] = syllableCount

print('Dictionary size:', len(syllableDict))


In [ ]:
def getSyllables(word):
    word = word.lower()
    if word in syllableDict:
        return syllableDict[word]
    else:
        return syllables.estimate(word)

def isComplexWord(word):
    if getSyllables(word) > complexThreshold:
        return True
    else:
        return False

print('abandon:', getSyllables('abandon'))
print('information:', getSyllables('information'))


In [ ]:
def isFuture(sentence):
    futureWords = ['will', 'shall', 'may', 'might', 'should']
    sentWords = word_tokenize(sentence.lower(), preserve_line=True)
    for word in futureWords:
        if word in sentWords:
            return True
    for i in range(len(sentWords) - 1):
        if sentWords[i] == 'going' and sentWords[i + 1] == 'to':
            return True
    return False

print(isFuture('We will expand.'))       # True
print(isFuture('We might expand.'))      # True
print(isFuture('We are going to grow.')) # True
print(isFuture('Goodwill increased.'))  # False

In [ ]:
tenKFiles = []
for fileName in sorted(os.listdir(downloadDir)):
    if fileName.lower().endswith('.txt') and os.path.isfile(os.path.join(downloadDir, fileName)):
        tenKFiles.append(fileName)
tenKFiles = tenKFiles[:100]
if len(tenKFiles) != 100:
    raise ValueError('Please check downloadDir: 100 TXT files are required.')

with open(os.path.join(wordListDir, 'PositiveWords.txt'), encoding='utf-8-sig') as f:
    positiveWords = set(f.read().lower().split())
with open(os.path.join(wordListDir, 'NegativeWords.txt'), encoding='utf-8-sig') as f:
    negativeWords = set(f.read().lower().split())
with open(os.path.join(wordListDir, 'LitigiousWords.txt'), encoding='utf-8-sig') as f:
    litigiousWords = set(f.read().lower().split())

print('10-K files:', len(tenKFiles))
print(tenKFiles[:5])


In [ ]:
def cleanText(raw):
    start = raw.upper().find('<DOCUMENT>')
    if start != -1:
        raw = raw[start + len('<DOCUMENT>'):]

        end = raw.upper().find('</DOCUMENT>')
        if end != -1:
            raw = raw[:end]

    # Keep the text section, excluding the SEC header.
    start = raw.upper().find('<TEXT>')
    if start != -1:
        raw = raw[start + len('<TEXT>'):]

        end = raw.upper().find('</TEXT>')
        if end != -1:
            raw = raw[:end]

    # Remove HTML code and hidden metadata.
    soup = BeautifulSoup(raw, 'html.parser')

    for tag in soup(['script', 'style', 'head', 'noscript',
                     'ix:header', 'ix:hidden']):
        tag.decompose()

    for tag in soup.find_all(True):
        if tag.parent is None:
            continue

        style = ''.join(tag.get('style', '').lower().split())
        if (tag.has_attr('hidden')
                or 'display:none' in style
                or 'visibility:hidden' in style):
            tag.decompose()

    cleaned = soup.get_text(separator=' ')
    return ' '.join(cleaned.split())


def getWords(text):
    textList = word_tokenize(text.lower(), preserve_line=True)
    words = []

    for word in textList:
        if word.isalpha():
            words.append(word)

    return words


In [ ]:
def getMetrics(words, sentenceCount):
    wordCount = len(words)
    positiveWordCount = 0
    negativeWordCount = 0
    litigiousWordCount = 0
    complexWordCount = 0

    for word in words:
        if word in positiveWords:
            positiveWordCount += 1
        if word in negativeWords:
            negativeWordCount += 1
        if word in litigiousWords:
            litigiousWordCount += 1
        if isComplexWord(word):
            complexWordCount += 1

    if wordCount > 0:
        positivityRate = positiveWordCount / wordCount
        negativityRate = negativeWordCount / wordCount
        litigiousRate = litigiousWordCount / wordCount
    else:
        positivityRate = float('nan')
        negativityRate = float('nan')
        litigiousRate = float('nan')

    if wordCount > 0 and sentenceCount > 0:
        fog = 0.4 * (wordCount / sentenceCount + 100 * complexWordCount / wordCount)
    else:
        fog = float('nan')

    return {
        'wordCount': wordCount,
        'PositiveWordCount': positiveWordCount,
        'NegativeWordCount': negativeWordCount,
        'LitigiousWordCount': litigiousWordCount,
        'PositivityRate': positivityRate,
        'NegativityRate': negativityRate,
        'LitigiousRate': litigiousRate,
        'sentenceCount': sentenceCount,
        'ComplexWordCount': complexWordCount,
        'FOG': fog
    }

In [ ]:
results = []
futureExamples = []
shortFiles = []

for count, fileName in enumerate(tenKFiles, start=1):
    with open(os.path.join(downloadDir, fileName), encoding='utf-8-sig', errors='replace') as tenK:
        raw = tenK.read()
    cleaned = cleanText(raw)
    sentences = sent_tokenize(cleaned)

    allWords = []
    futureWords = []
    futureSentenceCount = 0
    exampleCount = 0

    for sentence in sentences:
        words = getWords(sentence)
        allWords.extend(words)
        if isFuture(sentence):
            futureSentenceCount += 1
            futureWords.extend(words)
            if exampleCount < 3:
                futureExamples.append({'filename': fileName, 'sentence': sentence})
                exampleCount += 1

    wholeMetrics = getMetrics(allWords, len(sentences))
    futureMetrics = getMetrics(futureWords, futureSentenceCount)

    row = {'filename': fileName}
    for key in wholeMetrics:
        row[key] = wholeMetrics[key]
    for key in futureMetrics:
        row['future_' + key] = futureMetrics[key]
    results.append(row)

    if len(allWords) < 1000:
        shortFiles.append(fileName)
    print(count, fileName, 'words:', len(allWords),
          'sentences:', len(sentences), 'future sentences:', futureSentenceCount)

df = pd.DataFrame(results)
print('Dataframe shape:', df.shape)  # (100, 21)
df.head()
